# Doublets_4 Clustering - Spleen 55 only

Publication source copy for the Doublets 2 and 4 sequencing analyses. Outputs and execution counts are cleared. See `../VALIDATION_REPORT.md` for the separate execution results and remaining limitations.

Run from the `analysis/` directory after supplying the bundled inputs. Final figure sections explicitly load preserved manuscript checkpoints; they are distinct from fresh clustering.

In [ ]:
from pathlib import Path
Path("../results/Doublets_4_Clustering_Sp55").mkdir(parents=True, exist_ok=True)
from threadpoolctl import threadpool_limits
threadpool_limits(limits=12, user_api="blas")


In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
from anndata import read_h5ad
from anndata import read_text
from collections import Counter
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
from matplotlib.pyplot import rc_context
import anndata
import seaborn as sns
import regex as re
import os

In [ ]:
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=200, dpi_save=300)
sc.settings.n_jobs = 30
pd.options.display.max_columns = None
pd.options.display.max_rows = None

sc.logging.print_header()

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Import h5ad with processed data and BCR info

In [ ]:
path_data = '../data/Doublets/data/smartseq_doublets_4_analysis/'
adata = sc.read_h5ad(path_data+'Doublets4_7600cells_GeneSymbols_preprocessed_2xDoubletCounts.h5ad')

#the line below fixes an error that results when trying to run sc.tl.rank_genes_groups(adata,'leiden',method='wilcoxon')
#error is: KeyError: 'base'
adata.uns['log1p']["base"] = None
adata.obs.head()

# Filter interactions by imaging quality

In [ ]:
#filter our real doublets with low score; keep all artificial doublets and singlets
list_not_doubletscored = ['ArtDoub', 'Sing']
adata = adata[(adata.obs['sortedtype'].isin(list_not_doubletscored)) | (adata.obs['score']=='1')]
adata.shape

# Remove spheromers

In [ ]:
adata = adata[~((adata.obs['sortedpop'].str.contains('-HA')) | (adata.obs['sortedpop'].str.contains('-PRTN3')))]
adata.shape

# Subset 1 donor

In [ ]:
# Materialize the donor subset before PCA, as in the original analysis.
# Removing a scratch annotation otherwise leaves an AnnData view and changes
# PCA memory layout, which can change the subsequent Leiden partition.
adata = adata[adata.obs['SampleID']=='Spleen55'].copy()

adata.shape

In [ ]:
Counter(adata.obs['sortedpop'])

# PCA

In [ ]:
#compute pca
sc.tl.pca(adata, svd_solver='arpack')

In [ ]:
sc.pl.pca_scatter(adata, color=['SampleID'])#, save='_all_ages_droplet_facs_tissue.pdf')
sc.pl.pca(adata, color=['384plate', 'sortedtype', 'sortedpop'], wspace=0.5)
sc.pl.pca_scatter(adata, color=['n_counts', 'n_genes'])

# Cluster

In [ ]:
# Compute neighborhood graph followed by UMAP
sc.pp.neighbors(adata)
sc.tl.umap(adata)

#recommended method: Leiden graph clustering
sc.tl.leiden(adata,resolution=0.9)

In [ ]:
#plot
sc.pl.umap(adata, color=['leiden'], legend_fontsize=8)

sc.pl.umap(adata, color=['leiden','SampleID','Stimulation','sortedtype','sortedpop'], legend_fontsize=8)
df_grouped = adata.obs.groupby('leiden')['sortedpop'].value_counts(normalize=True).unstack()
df_grouped.plot.bar(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

df_grouped = adata.obs.groupby('sortedpop')['leiden'].value_counts(normalize=True).unstack()
df_grouped.plot.bar(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

# Rename leiden clusters

In [ ]:
sc.pl.umap(adata, color=['leiden','sortedpop','sortedtype'], legend_fontsize=8, wspace=0.4)

In [ ]:
# Numeric Leiden IDs can swap when equally sized clusters are reordered.
# Apply the published manual labels only after verifying exact cell membership.
cluster_annotations = pd.read_csv(
    '../data/annotations/Sp55_manuscript_cluster_annotations.tsv', sep='\t'
).sort_values('label_order', kind='stable')
computed_groups = {
    str(cluster_id): set(group['cell'])
    for cluster_id, group in adata.obs.groupby('leiden', observed=True)
}
cluster_labels = {}
for label, group in cluster_annotations.groupby('cluster_label', sort=False):
    expected_cells = set(group['cell'])
    matches = [cluster_id for cluster_id, cells in computed_groups.items()
               if cells == expected_cells]
    if len(matches) != 1:
        raise ValueError(
            f"Recomputed cluster membership differs for {label!r}; "
            "review the clustering before applying manuscript labels."
        )
    cluster_labels[matches[0]] = label

adata.obs['leiden_numeric'] = adata.obs['leiden'].astype(str)
labels = adata.obs['leiden_numeric'].map(cluster_labels)
keep = labels.notna()
label_order = cluster_annotations['cluster_label'].drop_duplicates().tolist()
adata.obs['leiden'] = pd.Categorical(labels, categories=label_order)
# The two unknown groups excluded in the original analysis have no manual labels.
adata = adata[keep].copy()


In [ ]:
adata_real = adata[adata.obs['sortedtype']=='RealDoub']
adata_real.shape

In [ ]:
Counter(adata.obs['sortedtype'])

# Plots

In [ ]:
sc.pl.umap(adata, color=['leiden','sortedpop','sortedtype'], legend_fontsize=8, wspace=0.4)
df_grouped = adata.obs.groupby('leiden')['sortedpop'].value_counts(normalize=True).unstack()
df_grouped.plot.barh(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

df_grouped = adata.obs.groupby('sortedpop')['leiden'].value_counts(normalize=True).unstack()
df_grouped.plot.barh(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

df_grouped = adata_real.obs.groupby('leiden')['Stimulation'].value_counts(normalize=True).unstack()
df_grouped.plot.barh(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

df_grouped = adata_real.obs.groupby('Stimulation')['leiden'].value_counts(normalize=True).unstack()
df_grouped.plot.barh(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

df_grouped = adata_real.obs.groupby('SampleID')['Stimulation'].value_counts(normalize=True).unstack()
df_grouped.plot.barh(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

In [ ]:
sc.pl.umap(adata, color=['leiden','AICDA'], legend_fontsize=8, cmap='YlGnBu')

# Find DEGs

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='leiden', method='logreg', tie_correct=True, pts=True, key_added='logreg')
cluster_genes = pd.DataFrame(adata.uns['logreg']['names'])
cluster_genes.head(100)

# Merge BCR data with RNA data

In [ ]:
df_bcr = pd.read_csv(path_data+'Doublets4_TCR_BCR_productive_completeVDJ_UniqueClones.txt', sep='\t')
print('df_bcr ',df_bcr.shape)

### Drop nan IGH ###
df_bcr.dropna(subset=['locus_h','locus_l'],inplace=True)
print('Drop nan IGH ',df_bcr.shape)
df_bcr.reset_index(inplace=True, drop=True)

### Drop rows that have an 'N' in their sequence alignment ####
df_bcr = df_bcr[~(df_bcr['sequence_alignment_h'].str.contains('N'))]
df_bcr.reset_index(inplace=True, drop=True)
df_bcr = df_bcr[~(df_bcr['sequence_alignment_l'].str.contains('N'))]
df_bcr.reset_index(inplace=True, drop=True)
print('Drop rows that have an N in their sequence alignment', df_bcr.shape)
df_bcr.head()

#### Merge adata with bcr #####
adata.obs = adata.obs.merge(df_bcr, on=['barcode'], how='left')
adata.obs.head()

In [ ]:
df_grouped = adata.obs.groupby('leiden')['c_call_h'].value_counts(normalize=True).unstack()
df_grouped.plot.barh(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

# Import TCR data and merge with RNA+BCR

In [ ]:
#import
path_data = '../data/Doublets/data/smartseq_doublets_4_analysis/'
df_ir = pd.read_csv(path_data+'Doublets4_TCR_BCR_productive_completeVDJ.txt', sep='\t')

#separate heavy and light chain
df_tcr_b = df_ir[df_ir['locus']=='TRB']
df_tcr_a = df_ir[df_ir['locus']=='TRA']

#keep only the first occurence of each TRB per donor
print(len(df_tcr_b))
df_tcr_b.sort_values(by=['v_score'],inplace=True,ascending=False)
df_tcr_b.drop_duplicates(subset=['cell_id'],inplace=True)
print(len(df_tcr_b))

#keep only the first occurence of each IgKL per donor
print(len(df_tcr_a))
df_tcr_a.sort_values(by=['v_score'],inplace=True,ascending=False)
df_tcr_a.drop_duplicates(subset=['cell_id'],inplace=True)
print(len(df_tcr_a))

#merge heavy and light chain
df_tcr = df_tcr_b.merge(df_tcr_a, on=['cell_id','barcode'], how='left', suffixes=('_b', '_a'))
print(df_tcr.shape)

### Drop nan TCR ###
df_tcr.dropna(subset=['locus_b','locus_a'],inplace=True)
print('Drop nan TCR ',df_tcr.shape)
df_tcr.reset_index(inplace=True, drop=True)

## Drop rows that have an 'N' in their sequence alignment ####

#drop columns that cause problem when saving adata (and is not useful)
df_tcr.dropna(axis=1, how='all',inplace=True)

#### Merge adata with bcr #####
adata.obs = adata.obs.merge(df_tcr, on=['barcode','cell_id'], how='left')
adata.obs.head()

# Add TCR and BCR expansion numbers

In [ ]:
###### Calculate clonal expansion by amino acid
adata.obs['junction_aa_h_count'] = [list(adata.obs['junction_aa_h']).count(x) if pd.notna(x) else np.nan for x in list(adata.obs['junction_aa_h'])]
adata.obs['junction_aa_h_expanded'] = ['Y' if x>1 else 'N' for x in list(adata.obs['junction_aa_h_count'])]

adata.obs['junction_aa_b_count'] = [list(adata.obs['junction_aa_b']).count(x) if pd.notna(x) else np.nan for x in list(adata.obs['junction_aa_b'])]
adata.obs['junction_aa_b_expanded'] = ['Y' if x>1 else 'N' for x in list(adata.obs['junction_aa_b_count'])]

In [ ]:
sc.pl.umap(adata, color=['leiden','sortedpop','junction_aa_h_count','junction_aa_h_expanded','junction_aa_b_count','junction_aa_b_expanded'], legend_fontsize=8, cmap='YlGnBu')

# Save

In [ ]:
#save processed data
adata.write_h5ad(path_data+'Doublets_SP55_2170cells_processed_2xDoubletCounts_BCR_TCR.h5ad')

# Manuscript figures 20250301

In [ ]:
# Use the preserved manuscript object for the published figure layout.
# Load h5ad

path_data = '../data/Doublets/data/smartseq_doublets_4_analysis/'
adata = sc.read_h5ad('../data/reference_checkpoints/Doublets_SP55_2170cells_processed_2xDoubletCounts_BCR_TCR.h5ad')

adata.uns['log1p']["base"] = None
adata

#rename leiden clusters
new_cluster_names = ['No Intx 1', 
                     'No Intx 2',
                     'Cytotoxic Treg - B Intx',
                     'Cytotoxic CD4/8 - B Intx',
                     'Treg/Tfr',
                     'Th1/Tfh - MZB Intx',
                     'Cytotoxic Treg - B Intx Cycling',
                     'Tfh',
                     'B',
                     'Treg/Teff',
                     'Tfh - GCB Intx',
                     'T - Plasmablast Intx']
adata.rename_categories('leiden', new_cluster_names)

#edit names

#merge cytotoxic Treg cycling and stable
dict_leiden_old_to_new = {'No Intx 1':'T-B non-interacting',
                          'No Intx 2':'T-B non-interacting',
                            'Cytotoxic Treg - B Intx':'Cytotoxic Treg-B',
                            'Cytotoxic CD4/8 - B Intx':'Cytotoxic CD4/8-B',
                            'Treg/Tfr':'Treg/Tfr',
                            'Th1/Tfh - MZB Intx':'Th1/Tfh-MZB',
                            'Cytotoxic Treg - B Intx Cycling':'Cytotoxic Treg-B',
                            'Tfh':'Tfh',
                            'B':'B',
                            'Treg/Teff':'Treg/Teff',
                            'Tfh - GCB Intx':'Tfh-GCB',
                            'T - Plasmablast Intx':'T-Plasmablast'}
adata.obs['leiden'] = (
    adata.obs['leiden']
    .map(dict_leiden_old_to_new)
    .astype('category')
)

#rename sortedpop categories

dict_sortedpop_old_to_new = {'CD25pTB': 'CD25+ T-B',
         'CD25nTB': 'CD25- T-B', 
         'ArtDoub-Tfr-B': 'Artificial Tfr-B', 
         'ArtDoub-Treg-B': 'Artificial Treg-B', 
         'ArtDoub-Tfh-B': 'Artificial Tfh-B', 
         'Treg': 'Treg',
         'Tfr': 'Tfr',
         'Tfh': 'Tfh',
         'B': 'B'}
adata.obs['sortedpop'] = (
    adata.obs['sortedpop']
    .map(dict_sortedpop_old_to_new)
    .astype('category')
)

#rename stimulation
dict_stimulation_old_to_new = {'NS':'Unstim',
                               'LAIV':'LAIV',
                               'HelaNuc':'Nuclear Lysate'}
adata.obs['Stimulation'] = (
    adata.obs['Stimulation']
    .map(dict_stimulation_old_to_new)
    .astype('category')
)

# Merge Change-O BCR annotations

In [ ]:
#add changeo to adata and plot

#load adata
path_data = '../data/Doublets/data/smartseq_doublets_4_analysis/'

# adata

#open changeo

df_changeo = pd.read_csv('../data/Doublets/data/smartseq_doublets_4_analysis/changeo/changeo_IGH_IGKL/allcells_outfmt_immcantation_db-pass_parse-select_clone-pass_germ-pass_Formatted_withSHM.tsv',sep='\t')

print('df_changeo',len(df_changeo))

#get barcode column
df_changeo['barcode'] = (
    df_changeo['sequence_id']
    .str.split(';').str[0]  # Take the part before the first semicolon
    .str.split('_').str[-2:]  # Take the last two elements split by underscore
    .str.join('-')  # Join them with a dash
)

#separate into IGH and IGKL
df_changeo_IGH = df_changeo[df_changeo['locus']=='IGH']
print('df_changeo_IGH',len(df_changeo_IGH))
df_changeo_IGKL = df_changeo[df_changeo['locus']!='IGH']
print('df_changeo_IGKL',len(df_changeo_IGKL))

#merge them with appropriate suffixes
df_changeo_merged = df_changeo_IGH.merge(df_changeo_IGKL, how='left', on=['barcode'],suffixes=['_changeo_h','_changeo_l'])
print('df_changeo_merged',len(df_changeo_merged))
df_changeo_merged.head()

#sort df_changeo_merged by support for V and J genes so highest scores are at top
df_changeo_merged.sort_values(by=['barcode','v_score_changeo_h','j_score_changeo_h','v_score_changeo_l','j_score_changeo_l'],inplace=True,ascending=False)

#drop duplicates of df_changeo_merged
df_changeo_merged.drop_duplicates(subset=['barcode'],inplace=True)
print('df_changeo_merged',len(df_changeo_merged))

#merge with adata
adata.obs = adata.obs.merge(df_changeo_merged, how='left', on=['barcode'])
adata.obs_names = adata.obs_names.astype(str)
print(adata.shape)

#### add clonal expansion data

#make clone
adata.obs['changeo_BCR_clone'] = adata.obs['junction_aa_changeo_h']+'-'+adata.obs['junction_aa_changeo_l']

#add counts
adata.obs['changeo_BCR_clone_count'] = adata.obs['changeo_BCR_clone'].map(adata.obs['changeo_BCR_clone'].value_counts())
adata.obs['clone_id_changeo_h_count'] = adata.obs['clone_id_changeo_h'].map(adata.obs['clone_id_changeo_h'].value_counts())
#filter

adata.obs.head()

In [ ]:
Counter(adata.obs['leiden'])
set(adata.obs['leiden'])

In [ ]:
path_figures = '../results/Doublets_4_Clustering_Sp55/'

sc.settings.figdir = path_figures
sc.set_figure_params(dpi=300, dpi_save=600)

leiden_order_1 = ['Cytotoxic CD4/8-B',
                'T-Plasmablast',
                'Th1/Tfh-MZB',
                
                'Cytotoxic Treg-B',
                'Tfh-GCB',
                'T-B non-interacting',
                'B',
                'Treg/Teff',
                'Treg/Tfr',
                'Tfh']
leiden_order_2 = leiden_order_1[::-1]

sortedpop_order_1 = ['CD25- T-B',
 'CD25+ T-B',
 'Artificial Tfr-B',
 'Artificial Treg-B',
 'Artificial Tfh-B',
 'Treg',
 'Tfr',
 'Tfh',
 'B']

sortedpop_order_2 = sortedpop_order_1[::-1]

In [ ]:
#color palette
leiden_palette_2 = [
    "#A1D99B",  # Light Green
    "#FED976",  # Yellow
    "#F4A3C3",  # Light Pink
    "#FD8D3C",  # Orange

    "#969696",  # Medium Gray
    "#31A354",  # Green
    "#B2182B",  # Dark Red
    
    "#3182BD",  # Blue
    "#BC80BD",  # Purple
    "#E6AB02"   # Goldenrod
]

## Figure 1

In [ ]:
#all 
adata_plot_leiden = adata.copy()
adata_plot_leiden.obs['leiden'] = adata_plot_leiden.obs['leiden'].cat.set_categories(leiden_order_2, ordered=True)

sc.pl.umap(adata_plot_leiden, color='leiden', legend_fontsize=12, frameon=False, title='', palette=leiden_palette_2, save='_Fig_2_SP55_leiden.pdf')

#singlets
adata_plot_leiden_singlet = adata.copy()
sortedpop_order_2_singlets = sortedpop_order_2[0:4]
adata_plot_leiden_singlet.obs['leiden'] = [x if y in sortedpop_order_2_singlets else 'none' for x,y in zip(adata_plot_leiden_singlet.obs['leiden'],adata_plot_leiden_singlet.obs['sortedpop'])] #mask to color leiden clusters by sorted population
adata_plot_leiden_singlet.obs['leiden'] = adata_plot_leiden_singlet.obs['leiden'].astype('category') #stop error
adata_plot_leiden_singlet.obs['leiden'] = adata_plot_leiden_singlet.obs['leiden'].cat.set_categories(leiden_order_2, ordered=True)
sc.pl.umap(adata_plot_leiden_singlet, color='leiden', legend_fontsize=12, frameon=False, title='Singlets', palette=leiden_palette_2, na_color='whitesmoke', legend_loc = None, save='_Fig_2_SP55_leiden_Singlets.pdf')

#artificial doublets
adata_plot_leiden_artdoub = adata.copy()
sortedpop_order_2_artdoub = sortedpop_order_2[4:7]
adata_plot_leiden_artdoub.obs['leiden'] = [x if y in sortedpop_order_2_artdoub else 'none' for x,y in zip(adata_plot_leiden_artdoub.obs['leiden'],adata_plot_leiden_artdoub.obs['sortedpop'])] #mask to color leiden clusters by sorted population
adata_plot_leiden_artdoub.obs['leiden'] = adata_plot_leiden_artdoub.obs['leiden'].astype('category') #stop error
adata_plot_leiden_artdoub.obs['leiden'] = adata_plot_leiden_artdoub.obs['leiden'].cat.set_categories(leiden_order_2, ordered=True)
sc.pl.umap(adata_plot_leiden_artdoub, color='leiden', legend_fontsize=12, frameon=False, title='Non-interacting Controls', palette=leiden_palette_2, na_color='whitesmoke', legend_loc = None, save='_Fig_2_SP55_leiden_ArtDoub.pdf')

#real doublets
adata_plot_leiden_realdoub = adata.copy()
sortedpop_order_2_realdoub = sortedpop_order_2[7:9]
adata_plot_leiden_realdoub.obs['leiden'] = [x if y in sortedpop_order_2_realdoub else 'none' for x,y in zip(adata_plot_leiden_realdoub.obs['leiden'],adata_plot_leiden_realdoub.obs['sortedpop'])] #mask to color leiden clusters by sorted population
adata_plot_leiden_realdoub.obs['leiden'] = adata_plot_leiden_realdoub.obs['leiden'].astype('category') #stop error
adata_plot_leiden_realdoub.obs['leiden'] = adata_plot_leiden_realdoub.obs['leiden'].cat.set_categories(leiden_order_2, ordered=True)
sc.pl.umap(adata_plot_leiden_realdoub, color='leiden', legend_fontsize=12, frameon=False, title='Contacting Pairs', palette=leiden_palette_2, na_color='whitesmoke', legend_loc = None, save='_Fig_2_SP55_leiden_RealDoub.pdf')

In [ ]:
#plot stim
adata_stim = adata.copy()

#select only true interactions
adata_stim = adata_stim[adata_stim.obs['sortedtype']=='RealDoub']
df_grouped = adata_stim.obs.groupby('Stimulation')['leiden'].value_counts(normalize=True).unstack()

df_grouped = df_grouped[leiden_order_2] #order columns

df_grouped.plot.barh(stacked=True,color=leiden_palette_2)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
sns.despine()
plt.savefig(path_figures+'barplot_Fig_2_Stimulation.pdf', bbox_inches='tight')

In [ ]:
## Plot distribution of sorted populations
df_grouped = adata.obs.groupby('sortedpop')['leiden'].value_counts(normalize=True).unstack()
df_grouped = df_grouped.reindex(sortedpop_order_2) #order rows
df_grouped = df_grouped[leiden_order_2] #order columns

#singlet

#singlet
df_grouped.loc[sortedpop_order_2[0:4]].plot.barh(stacked=True, color=leiden_palette_2, legend=False)
plt.grid(False)
plt.ylabel('')
sns.despine()
plt.savefig(path_figures+'barplot_Fig_2_singlet.pdf', bbox_inches='tight')
plt.show()

# artificial doublet
df_grouped.loc[sortedpop_order_2[4:7]].plot.barh(stacked=True, color=leiden_palette_2, legend=False)
plt.grid(False)
plt.ylabel('')
sns.despine()
plt.savefig(path_figures+'barplot_Fig_2_ArtDoub.pdf', bbox_inches='tight')
plt.show()

# real doublet
df_grouped.loc[sortedpop_order_1[0:2]].plot.barh(stacked=True, color=leiden_palette_2, legend=False)
plt.grid(False)
plt.ylabel('')
sns.despine()
plt.savefig(path_figures+'barplot_Fig_2_RealDoub.pdf', bbox_inches='tight')

plt.show()

In [ ]:
list_plot_genes = ['CD3E', 'MS4A1', 'FOXP3', 'IL2RA', 'TNFRSF18', 'PRF1', 'GZMA', 'GZMB', 'BCL6', 'AICDA', 'NRN1']

with plt.rc_context():  # Use this to set figure params like size and dpi
    #scaled
    sc.pl.dotplot(adata, list_plot_genes, 'leiden', dendrogram=False, cmap='Reds', standard_scale='var', colorbar_title='column scaled\nexpression',show=True, swap_axes=False, categories_order=leiden_order_2, save='Fig_2_SP55.pdf')

In [ ]:
list_plot_genes = ['CD3E','MS4A1',#lineage
                   'FOXP3','IL2RA','TNFRSF18',#Treg
                   'PRF1','GZMA','GZMB', #Cytotoxic
                   'BCL6','AICDA', #GC
                   'IL10','NRN1'
                   
                  ]

with plt.rc_context():  # Use this to set figure params like size and dpi
    #scaled
    sc.pl.dotplot(adata, list_plot_genes, 'leiden', dendrogram=False, cmap='Reds', standard_scale='var', colorbar_title='column scaled\nexpression',show=True, swap_axes=False, categories_order=leiden_order_2, 
                 )

## Figure S4

In [ ]:
list_plot_genes = ['CD3E', 'CD4', 'CD8A', 'MS4A1', 'FOXP3', 'IL2RA', 'TNFRSF18', 'PDCD1', 'PRF1', 'GZMA', 'GZMB', 'CXCR5', 'BCL6', 'BCL2A1', 'RGS13', 'AICDA', 'SOX5', 'ITGAX', 'CD1C', 'CD24', 'CXCR3', 'IL12A', 'IL12B', 'MZB1', 'JCHAIN', 'IFNG-AS1', 'KLRB1', 'GZMK', 'IL23A', 'IFNG', 'IFNG-AS1']

with plt.rc_context():  # Use this to set figure params like size and dpi
    #scaled
    sc.pl.dotplot(adata, list_plot_genes, 'leiden', dendrogram=False, cmap='Reds', standard_scale='var', colorbar_title='column scaled\nexpression',show=True, swap_axes=True, categories_order=leiden_order_2, save='Fig_S4_SP55.pdf')

In [ ]:
# Plot interactions by stimulation condition - ONLY realdoub 
adata_realdoublet = adata[adata.obs['sortedtype'].isin(['RealDoub'])]

df_grouped = adata_realdoublet.obs.groupby('Stimulation')['leiden'].value_counts(normalize=True).unstack()
df_grouped = df_grouped[leiden_order_2[4:]] #order columns

stimulation_palette = [
    
    "#B3DDF2",  # IGHD*00,IGHM*00 (similar light blue)
    "#FF7F00",  # IGHM*00 (distinct orange)
    "#6A3D9A",  # IGHG3*00 (dark purple)
    "#9E83AB",  # IGHG2*00 (medium purple)
    "#CAB2D6",  # IGHG1*00 (light purple)

     "#0B3D0B","#4C9B4C" #green
]

df_grouped.plot.barh(stacked=True, color=leiden_palette_2[4:])

plt.legend(
    bbox_to_anchor=(1.05, 0.5),  # Position the legend to the right of the figure
    loc="center left",           # Align the legend to the left vertically centered
    borderaxespad=0,
    frameon=False,
    fontsize=14,
    ncol=1                       # Single column
)
plt.grid(False)

plt.ylabel('')
plt.xlabel('% of cluster')
plt.savefig(path_figures+'barplot_Fig_S4_stimulation_leiden.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Plot interactions by stimulation condition - ONLY realdoub and only interaction clusters
adata_realdoublet = adata[(adata.obs['sortedtype'].isin(['RealDoub'])) & 
                        (adata.obs['leiden']!='T-B non-interacting')]

df_grouped = adata_realdoublet.obs.groupby('Stimulation')['leiden'].value_counts(normalize=True).unstack()
df_grouped = df_grouped[leiden_order_2[5:]] #order columns

stimulation_palette = [
    
    "#B3DDF2",  # IGHD*00,IGHM*00 (similar light blue)
    "#FF7F00",  # IGHM*00 (distinct orange)
    "#6A3D9A",  # IGHG3*00 (dark purple)
    "#9E83AB",  # IGHG2*00 (medium purple)
    "#CAB2D6",  # IGHG1*00 (light purple)

     "#0B3D0B","#4C9B4C" #green
]

df_grouped.plot.barh(stacked=True, color=leiden_palette_2[5:])

plt.legend(
    bbox_to_anchor=(1.05, 0.5),  # Position the legend to the right of the figure
    loc="center left",           # Align the legend to the left vertically centered
    borderaxespad=0,
    frameon=False,
    fontsize=14,
    ncol=1                       # Single column
)
plt.grid(False)

plt.ylabel('')
plt.xlabel('% of cluster')

plt.show()

In [ ]:
# Plot IG isotype distribution
df_grouped = adata.obs.groupby('leiden')['c_call_h'].value_counts(normalize=True).unstack()

df_grouped = df_grouped.rename(columns={'IGHD*00': 'IGHD', 
                                        'IGHD*00,IGHM*00': 'IGHD,IGHM',
                                        'IGHM*00':'IGHM', 
                                        'IGHG1*00':'IGHG1', 
                                        'IGHG2*00':'IGHG2', 
                                        'IGHG3*00':'IGHG3', 
                                        'IGHA1*00':'IGHA1', 
                                        'IGHA2*00':'IGHA2',})
df_grouped = df_grouped[['IGHD', 'IGHD,IGHM', 'IGHM', 'IGHG1', 'IGHG2', 'IGHG3', 'IGHA1', 'IGHA2']] #order columns

isotype_palette = [
    "#A6CEE3",  # IGHD*00 (light blue)
    "#B3DDF2",  # IGHD*00,IGHM*00 (similar light blue)
    "#FF7F00",  # IGHM*00 (distinct orange)
    "#6A3D9A",  # IGHG3*00 (dark purple)
    "#9E83AB",  # IGHG2*00 (medium purple)
    "#CAB2D6",  # IGHG1*00 (light purple)

     "#0B3D0B","#4C9B4C" #green
]

df_grouped.loc[leiden_order_1[:-3]].plot.barh(stacked=True, color=isotype_palette)

plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.ylabel('')
plt.xlabel('% of cluster')
plt.savefig(path_figures+'barplot_Fig_S4_leiden_isotype.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Compare each category to the reference category
from scipy.stats import ttest_ind

results = []
reference_category = 'B'
reference_group = list(adata[adata.obs['leiden'] == reference_category].obs['mu_freq_cdr_r_changeo_h'])
reference_group = [x for x in reference_group if pd.notna(x)]

for category in adata.obs['leiden'].unique():
    if category != reference_category:
        group = list(adata[adata.obs['leiden'] == category].obs['mu_freq_cdr_r_changeo_h'])
        #dropna
        group = [x for x in group if pd.notna(x)]

        t_stat, p_value = ttest_ind(reference_group, group, equal_var=False)  # Welch's t-test
        print(category,np.round(p_value,3))
        results.append({"Leiden": category, "t_stat": t_stat, "p_value": p_value})

In [ ]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Subset out T-only clusters
adata_noT = adata[~adata.obs['leiden'].isin(['Tfh', 'Treg/Tfr', 'Treg/Teff'])].copy()

# Plot order used in the violinplot
plot_order = leiden_order_2[3:]
palette_order = leiden_palette_2[3:]

# Make violinplot
plt.figure(figsize=(6, 8))
ax = sns.violinplot(
    data=adata_noT.obs,
    x="leiden",
    y="mu_freq_cdr_r_changeo_h",
    order=plot_order,
    palette=palette_order,
    inner='box'
)

plt.ylabel('IgH CDR SHM Frequency')
plt.xlabel('')
plt.xticks(rotation=90)
sns.despine()
plt.grid(False)

# ---------- Pairwise comparisons vs T-B non-interacting ----------
reference = 'T-B non-interacting'
exclude = {'B', reference}
comparisons = [grp for grp in plot_order if grp not in exclude]

ref_vals = adata_noT.obs.loc[
    adata_noT.obs['leiden'] == reference,
    'mu_freq_cdr_r_changeo_h'
].dropna()

raw_results = []
raw_pvals = []

for grp in comparisons:
    grp_vals = adata_noT.obs.loc[
        adata_noT.obs['leiden'] == grp,
        'mu_freq_cdr_r_changeo_h'
    ].dropna()

    stat, p = mannwhitneyu(grp_vals, ref_vals, alternative='two-sided')
    raw_results.append((grp, stat, p, len(grp_vals), len(ref_vals)))
    raw_pvals.append(p)

# Bonferroni correction
reject, pvals_bonf, _, _ = multipletests(raw_pvals, method='bonferroni')

# Print results
for (grp, stat, p_raw, n_grp, n_ref), p_adj, rej in zip(raw_results, pvals_bonf, reject):
    print(
        f"{grp} vs {reference}: "
        f"U={stat:.3f}, raw p={p_raw:.3e}, bonferroni p={p_adj:.3e}, "
        f"significant={rej}, n={n_grp} vs {n_ref}"
    )

# ---------- Convert adjusted p-values to stars ----------
def p_to_stars(p):
    if p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

# ---------- Annotate plot ----------
x_positions = {grp: i for i, grp in enumerate(plot_order)}
x_ref = x_positions[reference]

base_height = 0.31
step = 0.015
h = 0.008

for i, ((grp, stat, p_raw, n_grp, n_ref), p_adj) in enumerate(zip(raw_results, pvals_bonf)):
    x_grp = x_positions[grp]
    y = base_height + i * step
    stars = p_to_stars(p_adj)

    ax.plot([x_ref, x_ref, x_grp, x_grp], [y, y+h, y+h, y], lw=1.2, c='black')
    ax.text((x_ref + x_grp) / 2, y + h + 0.002, stars, ha='center', va='bottom', fontsize=11)

# ---------- Fixed limits AFTER annotation ----------
ax.set_ylim(-0.1, 0.4)
ax.set_yticks([0.0, 0.1, 0.2, 0.3, 0.4])  # don't label -0.1

plt.savefig(path_figures + 'violinplot_Fig_S4_sortedpop_shm.pdf', bbox_inches='tight')
plt.show()


# Calculate proportion expressing genes

In [ ]:
adata_subset = adata[adata.obs['leiden']=='Cytotoxic Treg-B']
adata_subset = adata_subset[adata_subset.obs['sortedpop']=='CD25+ T-B']

cytotoxic_cell_barcodes = list(set(list(adata_subset[adata_subset[: , 'PRF1'].X > 0, :].obs['barcode']) 
                                   + list(adata_subset[adata_subset[: , 'GZMA'].X > 0, :].obs['barcode']) 
                                   + list(adata_subset[adata_subset[: , 'GZMB'].X > 0, :].obs['barcode'])
                                  )
                              )

print('cytotoxic_cell_barcodes',len(cytotoxic_cell_barcodes)/len(adata_subset))

il10_cell_barcodes = list(set(list(adata_subset[adata_subset[: , 'IL10'].X > 0, :].obs['barcode'])))
print('il10_cell_barcodes',len(il10_cell_barcodes)/len(adata_subset))

cytotoxic_il10_cell_barcodes = list(set(list(adata_subset[adata_subset[: , 'PRF1'].X > 0, :].obs['barcode']) 
                                   + list(adata_subset[adata_subset[: , 'GZMA'].X > 0, :].obs['barcode']) 
                                   + list(adata_subset[adata_subset[: , 'GZMB'].X > 0, :].obs['barcode'])
                                   + list(adata_subset[adata_subset[: , 'IL10'].X > 0, :].obs['barcode'])
                                  )
                              )
print('cytotoxic_il10_cell_barcodes',len(cytotoxic_il10_cell_barcodes)/len(adata_subset))

neuritin_cell_barcodes = list(set(list(adata_subset[adata_subset[: , 'NRN1'].X > 0, :].obs['barcode'])))
print('neuritin_cell_barcodes',len(neuritin_cell_barcodes)/len(adata_subset))

In [ ]:
import numpy as np
from scipy import sparse

genes_off = ['PRF1','GZMA','GZMB']
gene_on   = 'IL10'

# 1) get the off‐matrix and on‐vector
X_off = adata_subset[:, genes_off].X
X_on  = adata_subset[:, gene_on].X

# 2) densify & flatten
if sparse.issparse(X_off):
    X_off = X_off.toarray()
if sparse.issparse(X_on):
    X_on = X_on.A1      # A1 flattens sparse to 1‑D
else:
    X_on = np.array(X_on).ravel()

# 3) build 1‑D masks
mask_off = (X_off <= 0).all(axis=1)   # shape (301,)
mask_on  =  X_on  >  0                # shape (301,)

# sanity check
assert mask_off.shape == mask_on.shape == (adata_subset.n_obs,)

# 4) combine
mask = mask_off & mask_on            # still (301,)

# 5) subset and pull barcodes
cytotoxic_il10_cell_barcodes = adata_subset[mask].obs['barcode'].tolist()

print(f"Found {len(cytotoxic_il10_cell_barcodes)} cells where PRF1/GZMA/GZMB ≤ 0 and IL10 > 0")

In [ ]:
sc.pl.scatter(adata_subset,x='PRF1',y='IL10',color='GZMA')

In [ ]:
list_plot_genes = ['CD1C','CD24','CD70','CR2','CD27','CXCL9','CXCL10','CXCR3','IL12A','IL12B','TNFSF4','TNFSF9','ITGAV','ITGB3']

with plt.rc_context():  # Use this to set figure params like size and dpi
    #scaled
    
    sc.pl.dotplot(adata, list_plot_genes, 'leiden', dendrogram=False, cmap='Blues', standard_scale='var', colorbar_title='column scaled\nexpression',show=True, swap_axes=True, categories_order=leiden_order_2)

In [ ]:
list_plot_genes = ['CD1C','CD24','CD70','CR2','CD27','CXCL9','CXCL10','CXCR3','IL12A','IL12B','TNFSF4','TNFSF9','ACKR3','GPR35']
list_plot_genes = ['CXCL9','CXCL10','CXCR5', 'CXCR3', 'CXCR6', 'CXCR2P1', 'CXCR2', 'CXCR1', 'CXCR4','CCR10',
 'CCRL2',
 'CCR7',
 'CCR2',
 'CCR5AS',
 'CCR1',
 'CCR5',
 'CCR9',
 'NCCRP1',
 'CCR12P',
 'CCR3',
 'CCR8',
 'CCR4',
 'CCR6','RARRES2','ACKR4P1', 'ACKR2', 'ACKR4', 'ACKR3']

with plt.rc_context():  # Use this to set figure params like size and dpi
    #scaled
    sc.pl.dotplot(adata, list_plot_genes, 'leiden', dendrogram=False, cmap='Blues', standard_scale='var', colorbar_title='column scaled\nexpression',show=True, swap_axes=True)

In [ ]:
list_plot_genes = ['CD3E','CD4','CD8A','MS4A1',#lineage
                   'FOXP3','IL2RA','TNFRSF18','PDCD1',#Treg
                   'PRF1','GZMA','GZMB', #Cytotoxic
                   'CD40LG','CXCR5','BCL6','MIR155HG','CCL22','EBI3','BCL2A1', #LZ
                   'AICDA','SOX5','ITGAX',#GC/ ABC B cells
                   'CD1C','CXCL9','CXCL10','CXCR3','ITGAV','ITGB3','CXCL13', #Th1-MZB
                   'MZB1','JCHAIN','SDC1','CD38','MYC','TFAP4','TNFRSF13B','IRF4','MKI67', #plasmablast
                   'BRCA1'
                   
                  ]

with plt.rc_context():  # Use this to set figure params like size and dpi
    #scaled
    sc.pl.dotplot(adata, list_plot_genes, 'leiden', dendrogram=False, cmap='Greys', standard_scale='var', colorbar_title='column scaled\nexpression',show=True, swap_axes=True, categories_order=leiden_order_2)

In [ ]:
# Plot IG isotype distribution
df_grouped = adata.obs.groupby('leiden')['c_call_h'].value_counts(normalize=True).unstack()
df_grouped = df_grouped[['IGHD*00','IGHD*00,IGHM*00','IGHM*00','IGHG1*00', 'IGHG2*00', 'IGHG3*00','IGHA1*00', 'IGHA2*00']] #order columns

isotype_palette = [
    "#A6CEE3",  # IGHD*00 (light blue)
    "#B3DDF2",  # IGHD*00,IGHM*00 (similar light blue)
    "#FF7F00",  # IGHM*00 (distinct orange)
    "#6A3D9A",  # IGHG3*00 (dark purple)
    "#9E83AB",  # IGHG2*00 (medium purple)
    "#CAB2D6",  # IGHG1*00 (light purple)
    "#FDBF6F",  # IGHA1*00 (light orange)
    "#FFEDA0"   # IGHA2*00 (light yellow-orange)
]

df_grouped.loc[leiden_order_1[:-3]].plot.barh(stacked=True, color=isotype_palette)

plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.ylabel('Annotated Leiden Cluster')
plt.xlabel('% of cluster')
plt.savefig(path_figures+'barplot_Fig_3_leiden_isotype.pdf', bbox_inches='tight')
plt.show()

In [ ]:
list_plot_genes = ['CD3E', 'CD4', 'CD8A', 'MS4A1', 'FOXP3', 'IL2RA', 'TNFRSF18', 'PDCD1', 'PRF1', 'GZMA', 'GZMB', 'CXCR5', 'BCL6', 'BCL2A1', 'RGS13', 'AICDA', 'SOX5', 'ITGAX', 'CD1C', 'CD24', 'CXCR3', 'IL12A', 'IL12B', 'MZB1', 'JCHAIN', 'IFNG-AS1', 'KLRB1', 'GZMK', 'IL23A', 'IFNG', 'IFNG-AS1']
with plt.rc_context():
    sc.pl.dotplot(adata, list_plot_genes, 'leiden', dendrogram=False, cmap='Reds', standard_scale='var', colorbar_title='column scaled\nexpression', show=True, swap_axes=False, categories_order=leiden_order_2, save='Fig_S4_SP55_swapaxes.pdf')